# Search Exploration

Interactive notebook for searching, browsing, and annotating papers.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display, Markdown

from src.search.arxiv_search import search_arxiv
from src.search.semantic_scholar_search import search_semantic_scholar
from src.search.unified_search import unified_search
from src.storage.paper_store import PaperStore, Paper
from src.analysis.tagger import apply_tags

store = PaperStore()
print(f"Papers in database: {store.count()}")

## Search

In [ ]:
# Run a search query
query = "GPU workload classification machine learning"
results = unified_search(query, max_results_per_source=10)
print(f"Found {len(results)} unique results")

In [ ]:
# Display results as a DataFrame
df = pd.DataFrame(results)
display(df[["title", "year", "source", "citation_count"]].head(20))

## Browse Abstracts

In [ ]:
# Preview abstracts for the top results
for i, paper in enumerate(results[:5]):
    title = paper.get("title", "Untitled")
    abstract = paper.get("abstract", "No abstract")[:300]
    year = paper.get("year", "?")
    display(Markdown(f"### {i+1}. {title} ({year})\n\n{abstract}...\n\n---"))

## Add Results to Database

In [ ]:
# Add search results to the paper store
added, skipped = store.merge_search_results(results)
print(f"Added: {added}, Skipped (duplicates): {skipped}")
print(f"Total papers: {store.count()}")

## Browse and Annotate

In [ ]:
# Browse all papers in the database
all_papers = store.get_all()
df_all = pd.DataFrame([
    {
        "id": p.id,
        "title": p.title,
        "year": p.year,
        "tags": ", ".join(p.tags),
        "status": p.read_status,
        "citations": p.citation_count,
    }
    for p in all_papers
])
display(df_all)

In [ ]:
# Annotate a specific paper (change paper_id to the ID you want to annotate)
# paper_id = "PAPER_ID_HERE"
# paper = store.get(paper_id)
# paper.notes = "Key findings: ..."
# paper.relevance_score = 0.9
# paper.read_status = "read"
# store.update(paper)